In [1]:
import Comparison_v4 as CP
import DataSetting_v5 as DS
import CompMethods_V3 as TP
#import TrainerVTS_V08F3 as TP

/home/caog/anaconda3/envs/cao3.8/lib/python3.8/site-packages/torchvision/io/image.py:11: UserWarning: Failed to load image Python extension: /home/caog/anaconda3/envs/cao3.8/lib/python3.8/site-packages/torchvision/image.so: undefined symbol: _ZNK3c1010TensorImpl36is_contiguous_nondefault_policy_implENS_12MemoryFormatE
  warn(f"Failed to load image Python extension: {e}")


In [2]:
env = 'B211'
name = env
data_path = [f"../dataset/Door_EXP/{env}"
            ]
level = 'subject'

data_organizer = DS.DataOrganizer(name, data_path, level)
data_organizer.load()

Cross validation plan at subject level
Loading ../dataset/Door_EXP/B211...

Loaded 0723D00-csi.npy of shape (1573034, 30, 3)
Loaded 0723D00-pd.npy of shape (1573034, 62)
Loaded 0723D10-csi.npy of shape (1263396, 30, 3)
Loaded 0723D10-pd.npy of shape (1263396, 62)
Loaded 0723D20-csi.npy of shape (1117300, 30, 3)
Loaded 0723D20-pd.npy of shape (1117300, 62)
Loaded 0723D30-csi.npy of shape (1076376, 30, 3)
Loaded 0723D30-pd.npy of shape (1076376, 62)
Loaded 0725D40-csi.npy of shape (1095983, 30, 3)
Loaded 0725D40-pd.npy of shape (1095983, 62)
Loaded 0725D50-csi.npy of shape (1425220, 30, 3)
Loaded 0725D50-pd.npy of shape (1425220, 62)
Loaded 0725D60-csi.npy of shape (1285295, 30, 3)
Loaded 0725D60-pd.npy of shape (1285295, 62)
Loaded 20240723_153414-bbx.npy of shape (6533, 4)
Loaded 20240723_153414-cimg.npy of shape (6533, 1, 128, 128)
Loaded 20240723_153414-ctr.npy of shape (6533, 2)
Loaded 20240723_153414-dpt.npy of shape (6533, 1)
Loaded 20240723_153414-rimg.npy of shape (6533, 128, 22

In [5]:
print(len(data_organizer.total_segment_labels))

49780


### Env package test

In [3]:
gpu = 1
preprocess = DS.Preprocess(new_size=(128, 128))
results = {}
for i in range(7):
    data_organizer.gen_plan()
    train_loader, valid_loader, test_loader, current_test = data_organizer.gen_loaders(mode='s', shuffle_test=False)
    if current_test == 'zhang2':
        continue
    
    model_path = f'../saved/Door_EXP/_COMP_sub_B211_{current_test}_TSVAE'
    # teacher_path = f'../saved//Door_EXP/_COMP_sub_A208_{current_test}_TSVAE/'

    trainer = TP.CompStudentTrainer(name='VAEStudent', mask=False, mode='vae_s',
                                                 networks=[TP.CSIEncoder(mode='tsvae', latent_dim=128, middle_dim=512*7*75), 
                                                           TP.ImageEncoder(mode='vae', latent_dim=128), 
                                                           TP.ImageDecoder(latent_dim=128)],
                                                lstm_steps = 75,
                                                 lr=1e-4, epochs=10, cuda=gpu,
                                                 preprocess = preprocess,
                                                 loss_optimizer = None,
                                                 train_loader=train_loader, valid_loader=valid_loader, test_loader=test_loader, notion=f'{name}_{current_test}_TSVAE')

    trainer.load(model_path, name='VAEStudent', mode='best')
    #trainer.load(teacher_path, name='VAETeacher', mode='best')
    trainer.test(loader='test')
    tester = CP.ResultCalculator(name='TSVAE', 
                    trainer=trainer, 
                    total_length = len(data_organizer.total_segment_labels)
                   )
    tester.fetch_preds()
    tester.evaluate(cuda=gpu)
    tester.save(f"TSVAE_{env}_{current_test}_Hist")
    results[current_test] = tester.results.mean()

Fetched level subject, 1 of 7, current = higashinaka
 Train set range = {'qiao', 'chen', 'zhang2', 'wang', 'jiao', 'zhang'}, len = 42710
 Test set current = higashinaka, len = 7070
Generating loaders for s: level = subject, current test = higashinaka
 Train dataset length = 42710
 Test dataset length = 7070
 Exported train loader of len 533, batch size = 64
 Exported valid loader of len 133, batch size = 64
 Exported test loader of len 111, batch size = 64

==========B211_higashinaka_TSVAE VAEStudent Loading==========
Loaded model CSIENCompV3 from VAEStudent_models_csien_best.pth!
Loaded model IMGENCompV3 from VAEStudent_models_imgen_best.pth!
Loaded model IMGDECompV3 from VAEStudent_models_imgde_best.pth!
==========B211_higashinaka_TSVAE VAEStudent Test starting==========



  0%|          |[00:00<?]


Test finished. Average loss={'LOSS': 2.4997988574139707, 'IMG': 27472.684966216217}

Total training time: 132.0281720161438 sec


Evaluating:   0%|          | 0/7070 [00:00<?, ?it/s]

Results saved
Fetched level subject, 2 of 7, current = chen
 Train set range = {'higashinaka', 'qiao', 'zhang2', 'wang', 'jiao', 'zhang'}, len = 41448
 Test set current = chen, len = 8332
Generating loaders for s: level = subject, current test = chen
 Train dataset length = 41448
 Test dataset length = 8332
 Exported train loader of len 518, batch size = 64
 Exported valid loader of len 129, batch size = 64
 Exported test loader of len 131, batch size = 64

==========B211_chen_TSVAE VAEStudent Loading==========
Loaded model CSIENCompV3 from VAEStudent_models_csien_best.pth!
Loaded model IMGENCompV3 from VAEStudent_models_imgen_best.pth!
Loaded model IMGDECompV3 from VAEStudent_models_imgde_best.pth!
==========B211_chen_TSVAE VAEStudent Test starting==========



...

Results saved
Fetched level subject, 7 of 7, current = zhang
 Train set range = {'higashinaka', 'qiao', 'chen', 'zhang2', 'wang', 'jiao'}, len = 40423
 Test set current = zhang, len = 9357
Generating loaders for s: level = subject, current test = zhang
 Train dataset length = 40423
 Test dataset length = 9357
 Exported train loader of len 505, batch size = 64
 Exported valid loader of len 126, batch size = 64
 Exported test loader of len 147, batch size = 64

==========B211_zhang_TSVAE VAEStudent Loading==========
Loaded model CSIENCompV3 from VAEStudent_models_csien_best.pth!
Loaded model IMGENCompV3 from VAEStudent_models_imgen_best.pth!
Loaded model IMGDECompV3 from VAEStudent_models_imgde_best.pth!
==========B211_zhang_TSVAE VAEStudent Test starting==========



  0%|          |[00:00<?]


Test finished. Average loss={'LOSS': 2.6359726174348066, 'IMG': 30678.72237723214}

Total training time: 112.4016969203949 sec


Evaluating:   0%|          | 0/9357 [00:00<?, ?it/s]

Results saved


In [4]:
for key, value in results.items():
    print(key, value)

higashinaka mse                       NaN
soft_iou                  NaN
matched_iou               NaN
matched_iou_mask          NaN
matched_mae               NaN
matched_mae_mask          NaN
average_depth_mse         NaN
est_depth                 NaN
gt_depth                  NaN
hist_mse             0.000369
distance                  NaN
dx                        NaN
dy                        NaN
dtype: float64
chen mse                       NaN
soft_iou                  NaN
matched_iou               NaN
matched_iou_mask          NaN
matched_mae               NaN
matched_mae_mask          NaN
average_depth_mse         NaN
est_depth                 NaN
gt_depth                  NaN
hist_mse             0.000391
distance                  NaN
dx                        NaN
dy                        NaN
dtype: float64
wang mse                       NaN
soft_iou                  NaN
matched_iou               NaN
matched_iou_mask          NaN
matched_mae               NaN
matched_mae_mask  

### Single test

In [25]:
gpu = 7
preprocess = DS.Preprocess(new_size=(128, 128))

data_organizer.gen_plan()
train_loader, valid_loader, test_loader, current_test = data_organizer.gen_loaders(mode='s', shuffle_test=False)

Fetched level subject, 6 of 6, current = wang
 Train set range = {'zhang', 'higashinaka', 'jiao', 'chen', 'qiao'}, len = 42841
 Test set current = wang, len = 6922
Generating loaders for s: level = subject, current test = wang
 Train dataset length = 42652
 Test dataset length = 6922
 Exported train loader of len 533, batch size = 64
 Exported valid loader of len 133, batch size = 64
 Exported test loader of len 109, batch size = 64



In [26]:
model_path = f'../saved/Door_EXP/_COMP_sub_A208_{current_test}_VAE'
# teacher_path = f'../saved//Door_EXP/20240925_Abaux_sub_{current_test}/'

trainer = TP.CompTrainer(name='VAE',
                      lstm_steps = 75,
                      beta=0.5,
                         mode='vae',
                      loss_optimizer = None,
                        networks=[TP.CSIEncoder(mode='vae', middle_dim=512*7*75, latent_dim=128), 
                                               TP.ImageDecoder(latent_dim=128)],
                            epochs=10, cuda=gpu,
                      preprocess = preprocess,
                      notion=f'{name}_{current_test}',
                      train_loader=train_loader, valid_loader=valid_loader, test_loader=test_loader,
                     )
trainer.load(model_path, name='VAE', mode='best')
#trainer.load(teacher_path, name='Teacher', mode='best')

==========Door_EXP/Test/_wang VAE Loading==========
Loaded model CSIENCompV3 from VAE_models_csien_best.pth!
Loaded model IMGDECompV3 from VAE_models_imgde_best.pth!


In [27]:
trainer.test(loader='test')
tester = CP.ResultCalculator(name='Prop', 
                trainer=trainer, 
                total_length = len(data_organizer.total_segment_labels)
               )
tester.fetch_preds()
tester.evaluate(cuda=1)

==========Door_EXP/Test/_wang VAE Test starting==========



  0%|          |[00:00<?]


Test finished. Average loss={'LOSS': 406.13152040254084}

Total training time: 21.671529531478882 sec


Evaluating:   0%|          | 0/6922 [00:00<?, ?it/s]

In [28]:
print(tester.results.mean())

mse                  0.021287
soft_iou             0.892540
matched_iou_mask     0.055390
matched_mae          0.106625
matched_mae_mask     0.729598
average_depth        0.295417
est_depth            0.531764
gt_depth             0.563227
distance            23.564301
dx                  -9.897573
dy                 -13.219590
dtype: float64
